# AutoML com Databricks - Classificação de Flores Iris

Este notebook demonstra como usar o **Databricks AutoML** para treinar automaticamente modelos de machine learning.

O AutoML irá:
* Preparar e processar os dados automaticamente
* Testar múltiplos algoritmos de classificação
* Otimizar hiperparâmetros
* Gerar um notebook com o código do melhor modelo
* Registrar todos os experimentos no MLflow

In [0]:
# Importar bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
import mlflow

# Carregar o dataset Iris do sklearn
iris = load_iris()

# Converter para DataFrame do pandas com nomes de colunas apropriados
df = pd.DataFrame(
    data=iris.data,
    columns=iris.feature_names
)

# Adicionar a coluna alvo (target)
df['target'] = iris.target

# Mapear os números das classes para nomes
target_names = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
df['target_name'] = df['target'].map(target_names)

print(f"Dataset carregado com {len(df)} amostras e {len(df.columns)} colunas")
print(f"\nPrimeiras linhas:")
display(df.head())

print(f"\nDistribuição das classes:")
print(df['target_name'].value_counts())

/databricks/python/lib/python3.11/site-packages/mlflow/protos/service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


Dataset carregado com 150 amostras e 6 colunas

Primeiras linhas:


sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,target_name
5.1,3.5,1.4,0.2,0,setosa
4.9,3.0,1.4,0.2,0,setosa
4.7,3.2,1.3,0.2,0,setosa
4.6,3.1,1.5,0.2,0,setosa
5.0,3.6,1.4,0.2,0,setosa



Distribuição das classes:
setosa        50
versicolor    50
virginica     50
Name: target_name, dtype: int64


## Preparação dos Dados

Agora vamos salvar os dados em uma tabela do Unity Catalog para que o AutoML possa acessá-los.

A tabela será criada em: `workspace.default.iris_data`

In [0]:
# Converter para Spark DataFrame
spark_df = spark.createDataFrame(df)

# Definir o nome da tabela no Unity Catalog
table_name = "workspace.default.iris_data"

# Salvar como tabela Delta no Unity Catalog
spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"✅ Tabela '{table_name}' criada com sucesso!")
print(f"\nVerificando a tabela:")

# Verificar a tabela criada
df_check = spark.table(table_name)
print(f"Registros na tabela: {df_check.count()}")
display(df_check.limit(5))

2026-06-04 14:40:07,153 3130 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py", line 1724, in _execute_and_fetch_as_iterator
    for b in generator:
  File "<frozen _collections_abc>", line 330, in __next__
  File "/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/reattach.py", line 139, in send
    if not self._has_next():
           ^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/reattach.py", line 200, in _has_next
    raise e
  File "/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/reattach.py", line 172, in _has_next
    self._current = self._call_iter(
                    ^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/reattach.py", line 297, in _call_iter
    raise e
  File "/databricks/python/lib/python3.11

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8010581778769416>, line 8
      5 table_name = "workspace.default.iris_data"
      7 # Salvar como tabela Delta no Unity Catalog
----> 8 spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
     10 print(f"✅ Tabela '{table_name}' criada com sucesso!")
     11 print(f"\nVerificando a tabela:")

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    711 self._write.table_name = name
    712 self._write.table_save_method = "save_as_table"
--> 713 _, _, ei = self._spark.client.execute_command(
    714     self._write.command(self._spark.client), self._write.observations
    715 )
    716 self._callback(ei)

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/client/c

## Executar Databricks AutoML

O AutoML irá:
1. Analisar os dados automaticamente
2. Testar diferentes algoritmos (Random Forest, XGBoost, LightGBM, etc.)
3. Fazer feature engineering automaticamente
4. Otimizar hiperparâmetros
5. Gerar notebooks com o código completo

**Parâmetros:**
* `target_col`: coluna alvo para prever (target)
* `timeout_minutes`: tempo máximo de execução (5 minutos)
* `max_trials`: número máximo de modelos a testar (10)

In [0]:
from databricks import automl

# Executar AutoML para classificação
print("🚀 Iniciando AutoML... Isso pode levar alguns minutos.\n")

summary = automl.classify(
    dataset=table_name,
    target_col="target",
    timeout_minutes=5,
    max_trials=10
)

print("\n✅ AutoML concluído!")
print(f"\nMelhor modelo: {summary.best_trial.model_description}")
print(f"Métrica de avaliação: {summary.metric}")
print(f"Melhor score: {summary.best_trial.metrics[summary.metric]:.4f}")

## Acessar e Usar o Melhor Modelo

O AutoML registra automaticamente o melhor modelo no MLflow. Podemos:
* Carregar o modelo para fazer previsões
* Ver os notebooks gerados com o código completo
* Acessar todos os experimentos no MLflow UI

In [0]:
# Obter informações do melhor modelo
best_run_id = summary.best_trial.mlflow_run_id
model_uri = f"runs:/{best_run_id}/model"

print(f"🏆 Melhor modelo:")
print(f"Run ID: {best_run_id}")
print(f"Model URI: {model_uri}")
print(f"\nNotebook gerado: {summary.best_trial.notebook_path}")

# Carregar o modelo
import mlflow.pyfunc
model = mlflow.pyfunc.load_model(model_uri)

print("\n📊 Fazendo previsões com o melhor modelo...\n")

# Preparar dados de teste (primeiras 5 amostras)
test_data = df[iris.feature_names].head(5)

# Fazer previsões
predictions = model.predict(test_data)

# Mostrar resultados
results_df = test_data.copy()
results_df['predicted_class'] = predictions
results_df['predicted_name'] = results_df['predicted_class'].map(target_names)
results_df['actual_class'] = df['target'].head(5).values
results_df['actual_name'] = df['target_name'].head(5).values

print("Comparação: Predições vs. Valores Reais")
display(results_df[['predicted_name', 'actual_name']])

# Verificar acurácia nas amostras de teste
accuracy = (results_df['predicted_class'] == results_df['actual_class']).mean()
print(f"\n✅ Acurácia nas amostras de teste: {accuracy * 100:.1f}%")